# 02 Experiments


## Section 2.1 - Setup


In [ ]:
import logging

import lakefs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

from helpers import (
    compute_binary_classification_metrics,
    create_branch,
    create_xgboost_classifier,
    deterministic_train_test_split,
    engineer_time_features,
    engineer_route_features,
    frequency_encode,
    compute_delay_rates,
    apply_delay_rates,
    lakefs_commit,
    load_metrics_json,
    load_predictions_parquet,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_precision_recall_curve,
    read_parquet,
    save_metrics_json,
    save_predictions_parquet,
    write_parquet,
)
from notebook_setup import build_notebook_config, initialize_lakefs_repository

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
LOGGER = logging.getLogger("02_experiments")


In [ ]:
config = build_notebook_config()
init_result = initialize_lakefs_repository(config)
print(init_result.message)

lakefs_client = lakefs.Client(
    host=config.endpoint,
    username=config.access_key,
    password=config.secret_key,
)

LOGGER.info("Loading silver dataset from branch-aware path")
silver_repo = lakefs.Repository(config.repo_name, client=lakefs_client)
silver_obj = silver_repo.branch("silver").object("silver/flights_2023_clean.parquet")
with silver_obj.reader(mode="rb") as reader:
    silver_df = pd.read_parquet(reader)

print("silver shape:", silver_df.shape)
display(silver_df.head())


## Section 2.2 - Experiment A: Time-Based Features

Engineer time-based features (hour, day-of-week, month, weekend, holiday, time bucket),
label-encode categoricals, train XGBoost, and persist gold artifacts to lakeFS.


In [ ]:
# --- 2.2a: Create experiment branch and engineer time features ---
BRANCH_TIME = "experiment-time-features"
create_branch(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    source_branch="silver",
)
print(f"Branch '{BRANCH_TIME}' ready (from silver)")

# Engineer time features from silver dataset
time_df = engineer_time_features(silver_df)
print("Time features engineered, shape:", time_df.shape)
display(time_df[["FL_DATE", "DEP_TIME", "hour_of_day", "day_of_week", "month",
                  "is_weekend", "is_holiday_period", "time_of_day_bucket"]].head(10))


In [ ]:
# --- 2.2b: Label-encode categoricals and build feature matrix ---

# Label-encode categorical columns for Experiment A
le_airline = LabelEncoder()
le_origin = LabelEncoder()
le_bucket = LabelEncoder()

time_df["airline_enc"] = le_airline.fit_transform(time_df["AIRLINE_CODE"].astype(str))
time_df["origin_enc"] = le_origin.fit_transform(time_df["ORIGIN"].astype(str))
time_df["bucket_enc"] = le_bucket.fit_transform(time_df["time_of_day_bucket"].astype(str))

# Define feature columns for Experiment A
TIME_FEATURE_COLS = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday_period",
    "bucket_enc",
    "airline_enc",
    "origin_enc",
    "DISTANCE",
]
TARGET_COL = "is_delayed"

features_time = time_df[TIME_FEATURE_COLS + [TARGET_COL]].copy()
print("Feature matrix shape:", features_time.shape)
print("Feature columns:", TIME_FEATURE_COLS)
print("Class balance:")
print(features_time[TARGET_COL].value_counts(normalize=True))


In [ ]:
# --- 2.2c: Split data and persist gold feature artifact ---
X_train_t, X_test_t, y_train_t, y_test_t = deterministic_train_test_split(
    features_time, target_col=TARGET_COL, test_size=0.2, random_state=42
)
print(f"Train: {X_train_t.shape}, Test: {X_test_t.shape}")
print(f"Train delay rate: {y_train_t.mean():.4f}, Test delay rate: {y_test_t.mean():.4f}")

# Write gold feature artifact to lakeFS
write_parquet(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    path="gold/features_time.parquet",
    df=features_time,
)
lakefs_commit(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    message="Gold layer: time-based features",
    metadata={"phase": "phase-5", "layer": "gold", "experiment": "time"},
)
print("gold/features_time.parquet committed on", BRANCH_TIME)


In [ ]:
# --- 2.2d: Train XGBoost model ---
model_time = create_xgboost_classifier(random_state=42)
model_time.fit(X_train_t, y_train_t)
print("XGBoost (time) training complete")

# Predict
y_pred_t = model_time.predict(X_test_t)
y_scores_t = model_time.predict_proba(X_test_t)[:, 1]

# Compute metrics
metrics_time = compute_binary_classification_metrics(y_test_t, y_pred_t, y_scores_t)
print("\n=== Experiment A (Time Features) Metrics ===")
for k, v in metrics_time.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# --- 2.2e: Generate and save evaluation visuals ---
cm_path = plot_confusion_matrix(y_test_t, y_pred_t, "confusion_matrix_time.png")
print("Saved:", cm_path)

pr_path = plot_precision_recall_curve(y_test_t, y_scores_t, "pr_curve_time.png")
print("Saved:", pr_path)

fi_names = list(X_train_t.columns)
fi_values = list(model_time.feature_importances_)
fi_path = plot_feature_importance(fi_names, fi_values, "feature_importance_time.png", top_n=15)
print("Saved:", fi_path)


In [ ]:
# --- 2.2f: Persist metrics + predictions to lakeFS and commit ---
import tempfile, os

# Save locally then upload
with tempfile.TemporaryDirectory() as tmpdir:
    # Metrics JSON
    metrics_local = os.path.join(tmpdir, "metrics_time.json")
    save_metrics_json(metrics_time, metrics_local)
    with open(metrics_local, "rb") as f:
        repo = lakefs.Repository(config.repo_name, client=lakefs_client)
        repo.branch(BRANCH_TIME).object("gold/metrics_time.json").upload(
            data=f.read(), mode="wb", content_type="application/json"
        )

    # Predictions parquet
    preds_df = pd.DataFrame({"y_true": y_test_t.values, "y_scores": y_scores_t})
    preds_local = os.path.join(tmpdir, "predictions_time.parquet")
    save_predictions_parquet(preds_df, preds_local)
    write_parquet(
        client=lakefs_client,
        repo_name=config.repo_name,
        branch_name=BRANCH_TIME,
        path="gold/predictions_time.parquet",
        df=preds_df,
    )

ref_time = lakefs_commit(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    message="Train XGBoost on time-based features, save metrics",
    metadata={"phase": "phase-5", "layer": "gold", "experiment": "time"},
)
print("Experiment A committed:", getattr(ref_time, "id", ref_time))
print("Metrics:", metrics_time)


## Section 2.3 - Experiment B: Route-Based Features

TODO: Engineer route-based features, train/evaluate model, save gold artifacts.


In [ ]:
# TODO: Implement Experiment B pipeline in Phase 6.


## Section 2.4 - Comparison & Merge

TODO: Compare metrics/predictions from both experiment branches and determine winner.


In [ ]:
# TODO: Implement comparison table, overlay PR curve, and merge decision in Phase 7.
